# Limpieza del dataframe netflix_titles.csv
## Junio 2025
## by Francis

### Importar funciones generales

In [1]:
import sys
sys.path.append(r"C:\Users\FRANCIS\Documents\Proyecto_Streaming\scripts") 
import utils
import pandas as pd

In [2]:
import importlib
import utils
importlib.reload(utils)

<module 'utils' from 'C:\\Users\\FRANCIS\\Documents\\Proyecto_Streaming\\scripts\\utils.py'>

### Carga del DataFrame

In [3]:
df_netflix = utils.cargar_datos(r"C:\Users\FRANCIS\Documents\Proyecto_Streaming\data\netflix_titles.csv")

### Inspección del DataFrame

In [148]:
df_netflix.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [149]:
df_netflix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


### Convertir columna con fechas a datetime 

In [4]:
df_netflix = utils.convertir_a_fecha(df_netflix, ['date_added'])

In [153]:
df_netflix.dtypes

show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
dtype: object

### Ver porcentaje de valores nulos

In [154]:
utils.resumen_nulos(df_netflix)

,cantidad_nulos,porcentaje_nulos
director,2634,29.908028
country,831,9.435676
cast,825,9.367549
date_added,98,1.112751
rating,4,0.045418
duration,3,0.034064
show_id,0,0.000000
type,0,0.000000
title,0,0.000000
release_year,0,0.000000


### Evaluamos en las columnas que no son importantes para los objetivos de los análisis y que tienen valores nulos 

In [ ]:
evaluar_director = utils.evaluar_fila(df_netflix, "director", ["date_added", "rating", "show_id","listed_in", "country", "type"])
evaluar_director.to_string() #sí hay suficientes nulos para eliminar filas (70%< es not null)

"     show_id     type                                                   title director                                                                                            cast country date_added  release_year rating   duration                                             listed_in                                                                                                                                            description\n5989   s5990    Movie  13TH: A Conversation with Oprah Winfrey & Ava DuVernay      NaN                                                                     Oprah Winfrey, Ava DuVernay     NaN 2017-01-26          2017    NaN     37 min                                                Movies  Oprah Winfrey sits down with director Ava DuVernay to discuss her Oscar-nominated film, historical cycles of oppression and the broken prison system.\n6763   s6764  TV Show                                         Fit for Fashion      NaN                                 

In [9]:
df_netflix = df_netflix.drop(evaluar_director.index)

In [10]:
evaluar_cast = utils.evaluar_fila(df_netflix, "cast", ["date_added", "rating", "show_id", "listed_in", "country", "type"])
evaluar_cast.to_string() #no hay suficientes nulos para eliminar filas (>70% es not null)

'Empty DataFrame\nColumns: [show_id, type, title, director, cast, country, date_added, release_year, rating, duration, listed_in, description]\nIndex: []'

In [11]:
evaluar_duration = utils.evaluar_fila(df_netflix, "duration", ["date_added", "rating", "show_id", "listed_in", "country", "type"])
evaluar_duration.to_string() #no hay suficientes nulos para eliminar filas (>70% es not null)

'Empty DataFrame\nColumns: [show_id, type, title, director, cast, country, date_added, release_year, rating, duration, listed_in, description]\nIndex: []'

### Reemplazamos en todas las columnas object del DataFrame los datos nulos por "Desconocido"

In [ ]:
columnas_object = ["director", "country", "rating", "duration", "cast"]
df_netflix[columnas_object] = df_netflix[columnas_object].fillna("Desconocido")

### Se convierte date_added a int para poder calcular la moda y mediana dentro de sus datos

In [13]:
df_netflix["date_added"] = df_netflix["date_added"].astype("int64")


In [14]:
df_netflix.dtypes

show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added       int64
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

### Se aplica la función que reemplaza los valores nulos en date_added por la mediana o moda de la agrupación de datos 

In [ ]:
df_netflix = utils.elegir_imputacion(df_netflix, columna="date_added", agrupacion=["listed_in", "type", "country"], umbral_dispersion=60)

### Se verifica si hay valores extraños o valores nulos después del reemplazo

In [ ]:
print(df_netflix["date_added"].unique())  
print(df_netflix["date_added"].isnull().sum())  

[1632528000000000000 1632441600000000000 1632355200000000000 ...
 1544054400000000000 1457481600000000000 1578700800000000000]
0


### Los valores están en formato int con nanosegundos: se hace la conversión a segundos con timestamp para convertir a datetime

In [ ]:
# Filtrar valores extremos antes de la conversión
df_netflix = df_netflix[df_netflix["date_added"] // 10**9 > 0]  
df_netflix = df_netflix[df_netflix["date_added"] // 10**9 < 1893456000]  

df_netflix["date_added"] = pd.to_datetime(df_netflix["date_added"] // 10**9, unit="s", errors="coerce")


### Se verifica aleatoriamente los datos, se confirma si hay valores nulos todavía y si la conversión a datetime fue realizada

In [ ]:
print(df_netflix[["date_added"]].sample(10))  
print(df_netflix["date_added"].isnull().sum())  
df_netflix.dtypes


### Se crean las columnas derivadas de date_added y se convierten a tipo int

In [ ]:
df_netflix["month_added"] = df_netflix["date_added"].dt.month.astype("Int64")  
df_netflix["year_added"] = df_netflix["date_added"].dt.year.astype("Int64")

In [20]:
print(df_netflix["date_added"].isnull().sum())  # ¿Quedan valores nulos después de todo?
print(df_netflix[["date_added", "month_added", "year_added"]].sample(10))  # Revisión rápida

0
     date_added  month_added  year_added
7539 2017-10-01           10        2017
2736 2020-04-01            4        2020
2914 2020-02-14            2        2020
8113 2018-09-05            9        2018
4078 2019-02-27            2        2019
2985 2020-01-27            1        2020
7616 2019-07-01            7        2019
3786 2019-06-01            6        2019
7172 2018-07-19            7        2018
8177 2018-04-06            4        2018


In [21]:
utils.resumen_nulos(df_netflix)

,cantidad_nulos,porcentaje_nulos
show_id,0,0.0
type,0,0.0
title,0,0.0
director,0,0.0
cast,0,0.0
country,0,0.0
date_added,0,0.0
release_year,0,0.0
rating,0,0.0
duration,0,0.0


### Datos duplicados

In [22]:
df_netflix.duplicated().sum() #NO HAY DATOS DUPLICADOS 

0

### Estandarizar los datos según la columna

In [23]:
df_netflix = utils.limpiar_texto(df_netflix, ["type", "country", "rating", "listed_in","title", "director", "cast","duration", "description"], conservar_mayusculas=["title", "director", "cast","duration", "description"]) 

In [24]:
df_netflix.head() 

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,month_added,year_added
0,s1,movie,Dick Johnson Is Dead,Kirsten Johnson,Desconocido,united states,2021-09-25,2020,pg-13,90 min,documentaries,"As her father nears the end of his life, filmm...",9,2021
1,s2,tv show,Blood & Water,Desconocido,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",south africa,2021-09-24,2021,tv-ma,2 Seasons,"international tv shows, tv dramas, tv mysteries","After crossing paths at a party, a Cape Town t...",9,2021
2,s3,tv show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",desconocido,2021-09-24,2021,tv-ma,1 Season,"crime tv shows, international tv shows, tv act...",To protect his family from a powerful drug lor...,9,2021
3,s4,tv show,Jailbirds New Orleans,Desconocido,Desconocido,desconocido,2021-09-24,2021,tv-ma,1 Season,"docuseries, reality tv","Feuds, flirtations and toilet talk go down amo...",9,2021
4,s5,tv show,Kota Factory,Desconocido,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",india,2021-09-24,2021,tv-ma,2 Seasons,"international tv shows, romantic tv shows, tv ...",In a city of coaching centers known to train I...,9,2021


In [25]:
df_netflix.dtypes

show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
month_added              Int64
year_added               Int64
dtype: object

### Guardar el DataFrame limpio

In [26]:
df_netflix.to_csv("netflix_limpio.csv", index=False)  
df_netflix.to_parquet("netflix_limpio.parquet")  